In [4]:
%pip install holidays
%pip install openmeteo-requests requests-cache retry-requests

Note: you may need to restart the kernel to use updated packages.
  Obtaining dependency information for openmeteo-requests from https://files.pythonhosted.org/packages/f9/ac/5fcc90394486247405ea5f12623de4bd93b4ab31a34e6b782ad1a9834d8c/openmeteo_requests-1.7.5-py3-none-any.whl.metadata
  Obtaining dependency information for requests-cache from https://files.pythonhosted.org/packages/a5/bf/c1775e49b350225bd851576ba75263bc728d8f05c0e31439a45f3429cc7b/requests_cache-1.3.3-py3-none-any.whl.metadata
  Obtaining dependency information for retry-requests from https://files.pythonhosted.org/packages/b1/f3/8ce908497bebbc2790ef06240a2c0fb28c096abb59062d88f85090464a5f/retry_requests-2.0.0-py3-none-any.whl.metadata
  Obtaining dependency information for niquests>=3.15.2 from https://files.pythonhosted.org/packages/65/51/eb07449eb9e7ecae4350b5a40bfcb002d6d26d58cf2cfd2f928ae7c2f559/niquests-3.21.2-py3-none-any.whl.metadata
  Obtaining dependency information for openmeteo-sdk>=1.22.0 from https://fil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 890.5/890.5 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 631.1/631.1 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 82.8 MB/s eta 0:00:00ta 0:00:01
  Attempting uninstall: idna
    Found existing installation: idna 3.4
    Uninstalling idna-3.4:
      Successfully uninstalled idna-3.4
  Attempting uninstall: attrs
    Found existing installation: attrs 22.1.0
    Uninstalling attrs-22.1.0:
      Successfully uninstalled attrs-22.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spyder 6.0.1 requires pyqt5<5.16,>=5.15, which is not installed.
spyder 6.0.1 requires pyqtwebengine<5.16,>=5.15, which is not installed.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import datetime as dt
import sys
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from utilities.preprocessor import *
from utilities.values import *

data_dir = '../data/'
model_dir = '../models/'

### 1. Merge the separate raw datafiles

In [ ]:
# combine all the raw files
raw_data_dir_paths = [data_dir + 'raw/2020', data_dir + 'raw/2021',]
full_dataset_df = read_all_files_in_dir(raw_data_dir_paths)

In [ ]:
# force starttime into datetime format so that it is usable
full_dataset_df['starttime'] = pd.to_datetime(full_dataset_df['starttime'])

# the Feb-Apr 2021 files are in UTC, while all earlier files are in New York local time, so convert them
is_utc = full_dataset_df['starttime'] >= '2021-02-01'
full_dataset_df.loc[is_utc, 'starttime'] = (
    full_dataset_df.loc[is_utc, 'starttime']
    .dt.tz_localize('UTC')
    .dt.tz_convert(NYC_TIMEZONE)
    .dt.tz_localize(None)
)

# remove duplicate trips (corrected Feb 2021 file overlaps with Jan 2021 files)
full_dataset_df = full_dataset_df.drop_duplicates(subset = ['bikeid', 'starttime', 'start station name'])

# sort by increasing start and end dates
full_dataset_df = full_dataset_df.sort_values(['starttime', 'stoptime'])

print('Total number of rows:', len(full_dataset_df))
print('Data period start:', full_dataset_df['starttime'].min())
print('Data period end:' , full_dataset_df['starttime'].max())

### 2. Preprocessing the dataset

#### 2a. Extract date features and get the rental counts for each station, date and time

In [ ]:
extractor = DateFeatureExtractor()

# remove the depot, which only has 1 rental and is not an actual station
full_dataset_df = full_dataset_df[full_dataset_df[STATION] != 'JCBS Depot']

# extract hour first, so that we can make sure that each station-date-hour has a value
full_dataset_df[DATE] = full_dataset_df['starttime'].apply(extractor.to_date)
full_dataset_df[HOUR_OF_DAY] = full_dataset_df['starttime'].apply(extractor.to_hour_of_day)

In [ ]:
# count rentals for each station-date-hour (only includes those with at least 1 rental)
rental_counts_df = full_dataset_df.groupby([STATION, DATE, HOUR_OF_DAY]).size().reset_index(name = NUM_RENTALS)

# make sure that every station-date-hour has a row, even if 0
stations = rental_counts_df[STATION].unique()
dates = pd.date_range('2020-01-01', '2021-04-30', freq = 'D').date
hours = range(24)

full_rental_counts_df = pd.MultiIndex.from_product(
    [stations, dates, hours], names = [STATION, DATE, HOUR_OF_DAY]
).to_frame(index = False)

# the last raw file ends at 19:59 on 30 Apr 2021 (New York time), so there is no data for 8pm-11pm that day
no_data = (full_rental_counts_df[DATE] == dt.date(2021, 4, 30)) & (full_rental_counts_df[HOUR_OF_DAY] >= 20)
full_rental_counts_df = full_rental_counts_df[~no_data]

In [ ]:
# merge the rental counts, fill rows with no values to 0
full_rental_counts_df = full_rental_counts_df.merge(rental_counts_df, on = [STATION, DATE, HOUR_OF_DAY], how = 'left')
full_rental_counts_df[NUM_RENTALS] = full_rental_counts_df[NUM_RENTALS].fillna(0).astype(int)

# merge the coordinates of each station
station_locations_df = full_dataset_df.groupby(STATION)[[LATITUDE, LONGITUDE]].first().reset_index()
full_rental_counts_df = full_rental_counts_df.merge(station_locations_df, on = STATION, how = 'left')

In [ ]:
# extract the other date features
full_rental_counts_df[WEEK_OF_YEAR] = full_rental_counts_df[DATE].apply(extractor.to_week_of_year)
full_rental_counts_df[SINE_WEEK_OF_YEAR] = extractor.to_sine_week_of_year(full_rental_counts_df[WEEK_OF_YEAR])
full_rental_counts_df[COSINE_WEEK_OF_YEAR] = extractor.to_cosine_week_of_year(full_rental_counts_df[WEEK_OF_YEAR])
full_rental_counts_df[DAY_OF_WEEK] = full_rental_counts_df[DATE].apply(extractor.to_day_of_week)
full_rental_counts_df[IS_PUBLIC_HOLIDAY] = full_rental_counts_df[DATE].apply(extractor.to_is_public_holiday)

# drop week of year, no longer needed
full_rental_counts_df = full_rental_counts_df.drop([WEEK_OF_YEAR], axis=1)

# sort for easy reading
full_rental_counts_df = full_rental_counts_df.sort_values([STATION, DATE, HOUR_OF_DAY])

In [ ]:
full_rental_counts_df.head()

#### 2b. Create new features using the actual rental counts of the previous hours

Since we are predicting the demand 1 hour ahead, we can use T-3 and T-2 hours. <br>
For example, if we are predicting the demand for 12pm at 11am, we can use the actual rental numbers for 9am and 10am 

In [ ]:
# timestamp of each row, so that the lags work across midnight
rental_timestamp = (
    pd.to_datetime(full_rental_counts_df[DATE])
    + pd.to_timedelta(full_rental_counts_df[HOUR_OF_DAY], unit = 'h')
)

# lookup table: rentals for each station at each hour
rentals_lookup = (
    full_rental_counts_df
    .assign(timestamp = rental_timestamp)
    .groupby([STATION, 'timestamp'])[NUM_RENTALS]
    .sum()
)
data_start = rental_timestamp.min()

for lag in [2, 3]:
    lag_timestamp = rental_timestamp - pd.Timedelta(hours = lag)
    lag_keys = pd.MultiIndex.from_arrays([full_rental_counts_df[STATION], lag_timestamp])

    # same station, exactly `lag` hours earlier; no row for that hour means 0 rentals
    lag_values = pd.Series(
        rentals_lookup.reindex(lag_keys).fillna(0).to_numpy(),
        index = full_rental_counts_df.index,
    )

    # hours before the dataset begins have no data, so set them to 0
    full_rental_counts_df[f'{NUM_RENTALS}_t-{lag}'] = lag_values.where(lag_timestamp >= data_start, 0)


# save to file
full_rental_counts_df.to_csv(data_dir + 'intermediate_processing/full_rental_counts.csv', index=False)
print('Total number of rows:', len(full_rental_counts_df))

In [ ]:
full_rental_counts_df.head()

#### 2c. Extract weather features

In [ ]:
# get the location and dates to query
weather_search_df = full_rental_counts_df[[STATION, LATITUDE, LONGITUDE, DATE]]
weather_search_df = weather_search_df.drop_duplicates()

start_date = weather_search_df[DATE].min() - dt.timedelta(days = 1)   # 1 extra day so the first hour has a T-1 weather value
end_date = weather_search_df[DATE].max()
stations = set(weather_search_df[STATION])

In [ ]:
# call the weather API for each location
extractor = WeatherFeatureExtractor(start_date, end_date)
weather_df = pd.DataFrame()

for i, station in enumerate(stations, start = 1):
    location_df = weather_search_df[weather_search_df[STATION] == station]

    # get historical weather data for the station
    station_weather_history_df = extractor.get_hourly_weather(station, location_df[LATITUDE].iloc[0], location_df[LONGITUDE].iloc[0])
    weather_df = pd.concat([weather_df, station_weather_history_df])

    # pause for 1 min after every 20 calls to stay under the API rate limit
    if i % 20 == 0:
        print(f"{i} stations done, sleeping for 60s...")
        time.sleep(60)

# save results
weather_df.to_csv(data_dir + 'intermediate_processing/historical_weather_features.csv', index=False)

In [ ]:
weather_df.head()

#### 2d. Merge the citibike and weather features

In [ ]:
# read previously stored intermediate files
full_rental_counts_df = pd.read_csv(data_dir + 'intermediate_processing/full_rental_counts.csv')
weather_df = pd.read_csv(data_dir + 'intermediate_processing/historical_weather_features.csv')

In [ ]:
merge_columns = [STATION, DATE, HOUR_OF_DAY]

# one weather row per station-date-hour
# drops the repeated hour on 1 Nov daylight savings end
weather_to_merge_df = (
    weather_df
    .drop(columns = [LATITUDE, LONGITUDE])
    .drop_duplicates(subset = merge_columns, keep = "first")
)

In [ ]:
# lag the weather by 1 hour: the weather recorded at hour T-1 is used to predict hour T
# done on a timestamp so that 11pm rolls over to 12am of the next date
lagged_timestamp = (
    pd.to_datetime(weather_to_merge_df[DATE])
    + pd.to_timedelta(weather_to_merge_df[HOUR_OF_DAY] + 1, unit = 'h')
)
weather_to_merge_df[DATE] = lagged_timestamp.dt.strftime('%Y-%m-%d')
weather_to_merge_df[HOUR_OF_DAY] = lagged_timestamp.dt.hour

# merge features
full_features_df = full_rental_counts_df.merge(
    weather_to_merge_df,
    on = merge_columns,
    how = "left",
    validate = "many_to_one",
)

# 2am doesn't exist on the days the clocks go forward, so 3am has no T-1 weather;
# fill it with the previous available hour (1am) for the same station
weather_cols = [col for col in weather_to_merge_df.columns if col not in merge_columns]
full_features_df = full_features_df.sort_values([STATION, DATE, HOUR_OF_DAY])
full_features_df[weather_cols] = full_features_df.groupby(STATION)[weather_cols].ffill()

# remove the 2am rows on the days the clocks go forward (that hour did not exist)
nonexistent_hour = full_features_df[DATE].isin(['2020-03-08', '2021-03-14']) & (full_features_df[HOUR_OF_DAY] == 2)
full_features_df = full_features_df[~nonexistent_hour]

In [ ]:
# drop columns that will not be used
full_features_df = full_features_df.drop(columns=[LATITUDE, LONGITUDE])

# save results
full_features_df.to_csv(data_dir + 'intermediate_processing/full_features.csv', index=False)

In [ ]:
full_features_df.head()

#### 2e. Feature selection and train-validation-test split

In [ ]:
full_features_df = pd.read_csv(data_dir + 'intermediate_processing/full_features.csv')

# drop the features that are no longer needed (refer to eda_weather_feature_selection.ipynb)
dropped_features = [RAIN, SNOWFALL, WIND_GUSTS, CLOUD_COVER_LOW, CLOUD_COVER_HIGH]
full_features_df = full_features_df.drop(dropped_features, axis=1)

In [ ]:
# split into the train-validation and test set
train_validation_df = full_features_df[full_features_df[DATE] <= "2021-02-28"]
test_df = full_features_df[full_features_df[DATE] >= "2021-03-01"]

In [ ]:
# keep categorical features as plain columns: each model encodes them in its own format
column_order = [DATE, NUM_RENTALS] + CATEGORICAL_COLUMNS + NUMERICAL_COLUMNS + OTHER_COLUMNS
save_to_parquet_file(train_validation_df[column_order], data_dir + 'processed/train_validation.parquet')
save_to_parquet_file(test_df[column_order], data_dir + 'processed/test.parquet')

print('Total number of rows in train-validation:', len(train_validation_df))
print('Total number of rows in test:', len(test_df))